In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv('../.env')

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("✅ OpenAI client ready")

✅ OpenAI client ready


In [2]:
LOAN_PRODUCTS = [
    {
        "name": "Personal Loan",
        "description": "Unsecured personal loan for any purpose",
        "min_income": 2500,
        "max_loan": 50,
        "requires_credit_history": False,
        "suitable_for": "Anyone with stable income"
    },
    {
        "name": "Home Improvement Loan",
        "description": "Loan specifically for property renovation",
        "min_income": 4000,
        "max_loan": 150,
        "requires_credit_history": True,
        "suitable_for": "Property owners with good credit"
    },
    {
        "name": "Education Loan",
        "description": "Loan for higher education expenses",
        "min_income": 2000,
        "max_loan": 80,
        "requires_credit_history": False,
        "suitable_for": "Students or parents funding education"
    },
    {
        "name": "Business Loan",
        "description": "Loan for self-employed business owners",
        "min_income": 5000,
        "max_loan": 200,
        "requires_credit_history": True,
        "suitable_for": "Self-employed applicants with good credit"
    },
    {
        "name": "Secured Loan",
        "description": "Loan backed by collateral for lower risk",
        "min_income": 1500,
        "max_loan": 100,
        "requires_credit_history": False,
        "suitable_for": "Applicants with poor credit but assets"
    }
]

print(f"✅ Loaded {len(LOAN_PRODUCTS)} loan products")

✅ Loaded 5 loan products


In [3]:
def get_eligible_products(applicant_data: dict) -> list:
    """
    Filters loan products based on hard eligibility rules.
    This runs BEFORE the LLM to reduce hallucination risk.
    
    Returns:
        list of eligible product dicts
    """
    eligible = []
    
    total_income = applicant_data.get('total_income', 0)
    credit_history = applicant_data.get('credit_history', 0)
    
    for product in LOAN_PRODUCTS:
        # Check income requirement
        if total_income < product['min_income']:
            continue
        
        # Check credit history requirement
        if product['requires_credit_history'] and credit_history != 1.0:
            continue
        
        eligible.append(product)
    
    return eligible


# Test it
sample_applicant = {
    "total_income": 6000,
    "credit_history": 1.0,
    "self_employed": 0
}

eligible = get_eligible_products(sample_applicant)
print(f"Eligible products for sample applicant: {len(eligible)}")
for p in eligible:
    print(f"  - {p['name']}")

Eligible products for sample applicant: 5
  - Personal Loan
  - Home Improvement Loan
  - Education Loan
  - Business Loan
  - Secured Loan


In [4]:
def generate_recommendations(applicant_data: dict, eligible_products: list, prediction: int) -> str:
    """
    Uses the LLM to write personalized recommendation text
    based on the rule-filtered eligible products.
    
    The LLM only writes — it does NOT decide eligibility.
    Eligibility was already decided by the rule engine in Cell 3.
    """
    if not eligible_products:
        return ""
    
    products_text = "\n".join([
        f"- {p['name']}: {p['description']} (suitable for: {p['suitable_for']})"
        for p in eligible_products
    ])
    
    decision_context = "approved for their requested loan" if prediction == 1 else "not approved for their requested loan at this time"
    
    prompt = f"""
You are a helpful financial advisor at LoanWise.
The applicant has been {decision_context}.

Based on their profile, they are eligible for the following products:
{products_text}

APPLICANT PROFILE:
- Total Monthly Income: ${applicant_data.get('total_income', 'N/A')}
- Credit History: {'Good' if applicant_data.get('credit_history') == 1.0 else 'Poor'}
- Employment: {'Self-Employed' if applicant_data.get('self_employed') == 1 else 'Salaried'}
- Education: {'Graduate' if applicant_data.get('education') == 1 else 'Not Graduate'}

Write a short, friendly paragraph (max 100 words) recommending these products.
- Mention each eligible product by name
- Explain briefly why each suits their profile
- Keep the tone warm and helpful
- Do NOT mention any rates, fees, or specific amounts
- End with "Contact us to learn more about any of these options."
- Write ONLY the recommendation paragraph, nothing else
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful financial advisor writing personalized product recommendations."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [5]:
def run_marketing_agent(applicant_data: dict, prediction: int) -> dict:
    """
    Full marketing agent pipeline:
    1. Filter eligible products by hard rules
    2. Generate personalized recommendation text via LLM
    3. Return recommendations to be appended to the email
    
    Parameters:
        applicant_data : dict of applicant features
        prediction     : 0 (denied) or 1 (approved)
    
    Returns:
        dict with eligible_products, recommendation_text, and product_count
    """
    # Step 1: Rule-based eligibility filter
    eligible_products = get_eligible_products(applicant_data)
    
    if not eligible_products:
        return {
            "eligible_products": [],
            "recommendation_text": "",
            "product_count": 0
        }
    
    # Step 2: LLM writes the recommendation
    recommendation_text = generate_recommendations(
        applicant_data, eligible_products, prediction
    )
    
    return {
        "eligible_products": [p['name'] for p in eligible_products],
        "recommendation_text": recommendation_text,
        "product_count": len(eligible_products)
    }

In [6]:
approved_applicant = {
    "name": "John Smith",
    "total_income": 7000,
    "credit_history": 1.0,
    "self_employed": 0,
    "education": 1,
    "loan_amount": 128,
    "loan_term": 360,
    "property_area": "Urban"
}

result = run_marketing_agent(approved_applicant, prediction=1)

print("=" * 50)
print(f"ELIGIBLE PRODUCTS ({result['product_count']}): {result['eligible_products']}")
print("=" * 50)
print("\nRECOMMENDATION TEXT:")
print("-" * 50)
print(result['recommendation_text'])

ELIGIBLE PRODUCTS (5): ['Personal Loan', 'Home Improvement Loan', 'Education Loan', 'Business Loan', 'Secured Loan']

RECOMMENDATION TEXT:
--------------------------------------------------
Congratulations on your loan approval! Based on your good credit and stable income, I recommend considering our Personal Loan for any personal needs, or perhaps the Home Improvement Loan if you’re looking to renovate your property. If you or your dependents are planning to pursue further studies, the Education Loan could be a great fit. Additionally, if you’re self-employed, our Business Loan might help you expand your ventures. Lastly, if you have any assets and find yourself with less-than-perfect credit in the future, our Secured Loan would be available to you as well. Contact us to learn more about any of these options.


In [7]:
denied_applicant = {
    "name": "Jane Doe",
    "total_income": 2000,
    "credit_history": 0.0,
    "self_employed": 1,
    "education": 0,
    "loan_amount": 200,
    "loan_term": 360,
    "property_area": "Rural"
}

result_denied = run_marketing_agent(denied_applicant, prediction=0)

print("=" * 50)
print(f"ELIGIBLE PRODUCTS ({result_denied['product_count']}): {result_denied['eligible_products']}")
print("=" * 50)
print("\nRECOMMENDATION TEXT:")
print("-" * 50)
print(result_denied['recommendation_text'])

ELIGIBLE PRODUCTS (2): ['Education Loan', 'Secured Loan']

RECOMMENDATION TEXT:
--------------------------------------------------
Thank you for considering LoanWise! Given your current profile, I recommend looking into an Education Loan, which can help you cover higher education expenses if you're looking to advance your skills or support someone in their educational journey. Also, a Secured Loan could be a great option for you, as it allows you to leverage any assets you may have to secure funding, making it more accessible despite your current credit situation. Both options could be beneficial based on your self-employed status and financial circumstances. Contact us to learn more about any of these options.


In [10]:
import sys
sys.path.append('../src')

from email_agent import generate_validated_email
from marketing_agent import run_marketing_agent

# Test the full combined email
approved_applicant = {
    "name": "John Smith",
    "total_income": 7000,
    "credit_history": 1.0,
    "self_employed": 0,
    "education": 1,
    "loan_amount": 128,
    "loan_term": 360,
    "property_area": "Urban"
}

email_result = generate_validated_email(approved_applicant, prediction=1)
marketing_result = run_marketing_agent(approved_applicant, prediction=1)

final_email = build_final_email(
    email_result['email'],
    marketing_result['recommendation_text']
)

print("COMPLETE FINAL EMAIL:")
print("=" * 60)
print(final_email)

COMPLETE FINAL EMAIL:
Dear John Smith,

We are pleased to inform you that your loan application for $128,000 has been approved. Your strong credit history and stable employment status have contributed to this positive decision.

We understand the importance of this loan to you and appreciate your trust in our institution. At this stage, we encourage you to review the terms and conditions associated with your approved loan. 

To proceed, please schedule an appointment with our office or respond to this email at your earliest convenience. Our team is here to guide you through the next steps to finalize your loan process. 

Thank you for choosing us for your financing needs. 

Sincerely,

The LoanWise AI Team

---

PERSONALIZED OFFERS FOR YOU:

Congratulations on your loan approval! Given your good credit and stable income, I highly recommend considering a Personal Loan for any personal needs or expenses. If you're a property owner looking to renovate, our Home Improvement Loan could be a